# DNS Reconnaissance

---

**MITRE ATT&CK** `T1590.002` · **Tactic:** Reconnaissance · **Difficulty:** Beginner · **Time:** ~45 min

> DNS is the internet's phone book. But unlike a phone book, it also tells you the org chart, the mail server, the CDN in use, the VPN endpoint, and sometimes the internal IP ranges. Before touching a single port, a smart attacker spends time reading what DNS gives away for free.

---

### What you'll understand after this notebook

1. What information each DNS record type (A, MX, NS, TXT, CNAME) leaks about an organization
2. How subdomain enumeration works — using wordlists to discover hidden infrastructure
3. How reverse DNS reveals internal hostnames and network structure
4. How defenders detect and limit DNS-based information leakage

---

### Real-world context

In 2019, the Capital One breach — 100 million records — began with SSRF (Server-Side Request Forgery) on a misconfigured WAF. The attacker identified the WAF subdomain through DNS enumeration. DNS reconnaissance is passive, generates no alerts on the target, and is completely legal to perform against any public domain. It's how every red team engagement starts.

## 1. DNS Record Types — What Each One Reveals

| Record | Purpose | Attacker intelligence value |
|--------|---------|----------------------------|
| **A** | Domain → IPv4 address | Direct server IP, hosting provider, CDN detection |
| **AAAA** | Domain → IPv6 address | IPv6 infrastructure (often less protected) |
| **MX** | Mail server | Email provider, anti-spam config, phishing target assessment |
| **NS** | Nameserver | DNS provider, potential zone transfer target |
| **TXT** | Arbitrary text | SPF records reveal mail infrastructure; DMARC gaps enable spoofing; sometimes API keys leak here |
| **CNAME** | Alias to another domain | Third-party services in use (Heroku, AWS, GitHub Pages) |
| **SOA** | Zone authority record | Admin email address (often a real person's email) |
| **PTR** | IP → hostname (reverse) | Internal naming conventions, server roles from hostnames |

### The TXT record problem

TXT records were designed for SPF/DMARC email authentication config. In practice, they also expose:
- `google-site-verification=` — confirms Google Workspace use
- `MS=ms...` — confirms Office 365 tenant
- `v=spf1 include:sendgrid.net` — email provider is SendGrid
- Occasionally: accidentally committed API keys or tokens

In [ ]:
import dns.resolver       # dnspython — the standard DNS library for Python
import dns.reversename    # for PTR/reverse DNS lookups
import socket             # for gethostbyaddr (alternative reverse DNS method)
import concurrent.futures # for parallel subdomain enumeration
import matplotlib.pyplot as plt
import networkx as nx     # for domain tree visualization
from pprint import pprint

# Test the resolver is working
test = dns.resolver.resolve('google.com', 'A')
print(f'DNS resolver working. google.com → {[str(r) for r in test]}')

## 2. Querying All Record Types

The original script only queries A records. Let's build a comprehensive record harvester that extracts everything useful.

In [ ]:
RECORD_TYPES = ['A', 'AAAA', 'MX', 'NS', 'TXT', 'CNAME', 'SOA']

def harvest_records(domain):
    """
    Queries all common DNS record types for a domain.
    Returns a dict of {record_type: [results]}.
    """
    records = {}
    
    for rtype in RECORD_TYPES:
        try:
            answers = dns.resolver.resolve(domain, rtype)
            records[rtype] = [str(r) for r in answers]
        except dns.resolver.NoAnswer:
            # This record type doesn't exist for this domain — expected
            pass
        except dns.resolver.NXDOMAIN:
            # Domain doesn't exist at all
            print(f'[!] {domain} — NXDOMAIN (domain does not exist)')
            return {}
        except dns.exception.Timeout:
            print(f'[!] Timeout querying {rtype} for {domain}')
    
    return records


def print_records(domain, records):
    print(f'\n DNS Records for {domain}')
    print('═' * 50)
    for rtype, values in records.items():
        print(f'\n  {rtype}')
        for v in values:
            # Truncate long TXT records for readability
            display = v[:80] + '...' if len(v) > 80 else v
            print(f'    {display}')


# Run against a public domain
domain = 'github.com'
records = harvest_records(domain)
print_records(domain, records)

## 3. Reverse DNS — From IP to Hostname

The original script's `ReverseDNS()` function does a PTR lookup. This is powerful post-scan: after finding an IP is open, reverse DNS often reveals its role (`db01.internal.corp.com`, `vpn.company.com`, etc.).

In [ ]:
def reverse_dns(ip):
    """
    PTR record lookup — maps an IP back to its hostname(s).
    Returns a list of hostnames, or empty list if no PTR record.
    """
    try:
        # socket.gethostbyaddr returns (hostname, aliases, addresses)
        result = socket.gethostbyaddr(ip)
        return [result[0]] + result[1]  # primary + aliases
    except socket.herror:
        return []  # No PTR record configured
    except socket.gaierror:
        return []  # General DNS failure


# Demonstrate on a few known IPs
test_ips = ['8.8.8.8', '1.1.1.1', '208.67.222.222']

print(f'{'IP Address':<20} Hostname')
print('─' * 50)
for ip in test_ips:
    hostnames = reverse_dns(ip)
    print(f'{ip:<20} {hostnames[0] if hostnames else "(no PTR record)"}')

## 4. Subdomain Enumeration

The most valuable DNS technique. Organizations have hundreds of subdomains — dev, staging, admin portals, internal tools accidentally exposed. The original script uses a wordlist file. We'll build an in-memory version for the notebook.

**Common high-value subdomains attackers look for:**
- `vpn.`, `remote.`, `citrix.` — remote access
- `admin.`, `portal.`, `dashboard.` — privileged interfaces
- `api.`, `api-dev.`, `staging-api.` — API endpoints
- `mail.`, `outlook.`, `webmail.` — email infrastructure
- `dev.`, `staging.`, `test.`, `uat.` — less-secured environments
- `jenkins.`, `gitlab.`, `jira.`, `confluence.` — DevOps tools
- `s3.`, `bucket.`, `assets.` — storage infrastructure

In [ ]:
# Condensed wordlist for notebook demo (real lists have 10k+ entries)
SUBDOMAIN_WORDLIST = [
    'www', 'mail', 'remote', 'vpn', 'api', 'dev', 'staging', 'test',
    'admin', 'portal', 'dashboard', 'blog', 'docs', 'help', 'support',
    'cdn', 'static', 'assets', 'media', 'img', 'images',
    'git', 'gitlab', 'github', 'jenkins', 'ci', 'jira', 'confluence',
    'ftp', 'smtp', 'pop', 'imap', 'ns1', 'ns2', 'mx', 'mx1',
    'internal', 'intranet', 'corp', 'office',
]


def check_subdomain(subdomain, domain):
    """Returns (fqdn, ip_list) if subdomain exists, else None."""
    fqdn = f'{subdomain}.{domain}'
    try:
        result = dns.resolver.resolve(fqdn, 'A', lifetime=2)
        ips = [str(r) for r in result]
        return (fqdn, ips)
    except (dns.resolver.NXDOMAIN, dns.resolver.NoAnswer, dns.exception.Timeout):
        return None


def subdomain_search(domain, wordlist, max_workers=20):
    """
    Parallel subdomain enumeration using a thread pool.
    max_workers controls concurrency — higher = faster but noisier.
    """
    found = []
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_sub = {
            executor.submit(check_subdomain, sub, domain): sub
            for sub in wordlist
        }
        
        for future in concurrent.futures.as_completed(future_to_sub):
            result = future.result()
            if result:
                fqdn, ips = result
                found.append({'fqdn': fqdn, 'ips': ips})
                print(f'  ● {fqdn:<45} {ips[0]}')
    
    return sorted(found, key=lambda x: x['fqdn'])


print(f'Subdomain enumeration — github.com\n')
print(f'  {'FQDN':<45} IP')
print(f'  {"─" * 60}')
found_subdomains = subdomain_search('github.com', SUBDOMAIN_WORDLIST)
print(f'\n  {len(found_subdomains)} subdomains discovered.')

## 5. Visualizing the Domain Tree

A graph makes the scope of an organization's DNS footprint immediately obvious. Nodes colored by IP prefix show which subdomains share infrastructure.

In [ ]:
G = nx.DiGraph()

# Root domain as center node
root = 'github.com'
G.add_node(root, node_type='root')

# Build color map by /24 IP prefix (same prefix = same color = same datacenter)
prefix_colors = {}
color_palette = ['#6C9BCF', '#CF9B6C', '#9B6CCF', '#6CCF9B', '#CFB86C', '#CF6CA0']

def get_prefix_color(ip):
    prefix = '.'.join(ip.split('.')[:3])
    if prefix not in prefix_colors:
        prefix_colors[prefix] = color_palette[len(prefix_colors) % len(color_palette)]
    return prefix_colors[prefix]

node_colors = ['#CF6C6C']  # root is red

for entry in found_subdomains:
    fqdn = entry['fqdn']
    ip = entry['ips'][0] if entry['ips'] else '?'
    G.add_node(fqdn, node_type='subdomain', ip=ip)
    G.add_edge(root, fqdn)
    node_colors.append(get_prefix_color(ip))

if len(G.nodes) > 1:
    fig, ax = plt.subplots(figsize=(14, 8))
    fig.patch.set_facecolor('#0F0F0F')
    ax.set_facecolor('#0F0F0F')

    pos = nx.spring_layout(G, seed=42, k=2)
    nx.draw_networkx(
        G, pos, ax=ax,
        node_color=node_colors,
        node_size=800,
        edge_color='#282828',
        font_color='#EDEAE4',
        font_size=7,
        arrows=True,
        arrowsize=10
    )

    ax.set_title(f'DNS Subdomain Graph — {root}', color='#EDEAE4', fontsize=13)
    plt.tight_layout()
    plt.savefig('../cyberlab/content/articles/02_dns_graph.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No subdomains found to graph. Try a larger wordlist.')

## 6. Defender's Perspective

### What you can't stop

DNS is public by design. You cannot prevent someone from querying your public DNS records. What you *can* do:

- **Disable DNS zone transfers** to unauthorized resolvers (`allow-transfer { none; };` in BIND config). Zone transfers return the entire DNS zone in one response — a complete map of all records.
- **Audit your TXT records** — remove anything that reveals your internal tooling or has expired verification tokens.
- **Wildcard DNS** can obscure which subdomains exist, but breaks most enumeration defenses — attackers can still probe.
- **Separate internal DNS** — internal hostnames should never be in public DNS. Use split-horizon DNS to serve different records internally vs externally.

### What to audit in your own DNS

| Check | Risk |
|-------|------|
| `dig axfr @ns1.yourdomain.com yourdomain.com` | Zone transfer enabled = full DNS dump |
| TXT records containing internal IPs | Leaks internal network structure |
| Subdomains pointing to decommissioned cloud services | Subdomain takeover vulnerability |
| MX records with missing SPF | Open relay, spoofing risk |
| DMARC `p=none` | Email spoofing not prevented |

## 7. Article Seed

---

**Suggested title:** *What Your DNS Records Reveal Before You're Even Hacked*

**Opening paragraph (edit this):**

> DNS reconnaissance requires no special access, generates no alerts, and is completely legal. Before a single port is scanned, an attacker can learn which cloud providers you use, who handles your email, which third-party services you've integrated, and often find subdomains pointing at infrastructure you've forgotten about. In this article, I'll show you exactly what that looks like in Python — and what you should audit before someone else does.

**Section headers:**
1. What each DNS record type leaks (and why TXT records are the most dangerous)
2. Subdomain enumeration in Python — parallel DNS bruteforcing
3. The subdomain takeover problem — abandoned records pointing at live attackers
4. Auditing your own DNS before attackers do

**Medium tags:** `cybersecurity, dns, python, osint, recon`

---

In [ ]:
# Self-check

# Q1: Which DNS record type maps an IP back to a hostname?
reverse_record_type = None  # 'A', 'MX', 'PTR', or 'TXT'?
assert reverse_record_type == 'PTR', 'Reverse DNS uses PTR records'

# Q2: What DNS feature returns the complete zone contents to any requester?
zone_leak_feature = None  # answer as a string
assert 'transfer' in zone_leak_feature.lower(), 'Zone transfers — disable with allow-transfer { none; }'

# Q3: What does NXDOMAIN mean?
nxdomain_means = None  # answer as a string
assert 'exist' in nxdomain_means.lower(), 'Non-Existent Domain — the queried name has no DNS records'

print('All checks passed. You understand DNS reconnaissance.')